# UPI-Based Credit Scoring System

**PROBLEM:** A lot of Indians do not have a formal credit score, because they don't take formal loans. However they do use UPI, which could give a good indicator about their spending habits and hence can be a good estimator of reliability for credit scores.

**Key steps:**
- 1 - Simulate realistic UPI transaction data for 70000 users
- 2 - Create 58 behavioural features from raw transactions
- 3 - Create proxy default labels from stress signals
- 4 - Train logistic regression + XGBoost 
- 5 - Output credit scoresin a CIBIL format (300–900)
- 6 - Visualization

> The data has to be synthetic as real UPI transactions are private and cannot be referenced

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import warnings
from datetime import datetime, timedelta
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    classification_report, roc_auc_score,
    confusion_matrix, ConfusionMatrixDisplay, roc_curve
)
from sklearn.ensemble import RandomForestClassifier

try:
    import xgboost as xgb
    USE_XGB = True
    print("✅ XGBoost found — using XGBoost for Model B")
except ImportError:
    USE_XGB = False
    print("⚠️  XGBoost not found — falling back to Random Forest")
    print("   Fix: pip install xgboost")

warnings.filterwarnings("ignore")
np.random.seed(42)
import random
random.seed(42)

⚠️  XGBoost not found — falling back to Random Forest
   Fix: pip install xgboost


---
## 1. Data Simulation

No real data is available for UPI transactions, hence will be modelling archetypes, a characteristic that a user can have 

### archetypes:

| archetype | income range | default risk | inference  |
|---|---|---|---|
| salaried | ₹25k–80k/mo | 5% | fixed salary- predictable and regular |
| gig worker | ₹8k–40k/mo | 18% | income is not stable |
| small business | ₹15k–2L/mo | 12% | could range from lows to highs, high variance |
| student | ₹0–10k/mo | 25% | low income, high spending |
| rural Farmer | ₹5k–30k/mo | 20% | income is seasonal and irregular |

each archetype has:
- `income_range`: min/max monthly income
- `default_risk`: base probability of financial stress
- `spending_ratio`: what fraction of income they typically spend
- `bill_regularity`: how consistently they pay bills
- `gig_variance`: how much their income jumps month to month

In [10]:
ARCHETYPES = {
    "salaried_stable": {
        "income_range":(25000, 80000),
        "default_risk":0.05,
        "spending_ratio":(0.6, 0.8),
        "bill_regularity":0.95,
        "gig_variance":0.05,
    },
    "gig_worker": {
        "income_range":(8000, 40000),
        "default_risk":0.18,
        "spending_ratio":(0.7, 0.95),
        "bill_regularity":0.75,
        "gig_variance":0.40,           #income is all over the place
    },
    "small_business": {
        "income_range":(15000, 200000),
        "default_risk":0.12,
        "spending_ratio":(0.5, 0.9),
        "bill_regularity":0.85,
        "gig_variance":0.30,
    },
    "student": {
        "income_range":(0, 10000),
        "default_risk":0.25,
        "spending_ratio":(0.8, 1.2),   #could spend more than income if sent money by family
        "bill_regularity":0.60,
        "gig_variance":0.50,
    },
    "rural_farmer": {
        "income_range":(5000, 30000),
        "default_risk":0.20,
        "spending_ratio":(0.6, 0.85),
        "bill_regularity":0.70,
        "gig_variance":0.60,            #highly seasonal
    },
}

print("archetypes defined")

archetypes defined


### 1.1. Reference Data
some references explained:
- INDIAN_CITIES: a subset of the cities for these transaction
- CITY_TIER: the tier of the city, based on cost of living, the high COLs are listed as tier 1 and moves down the list
- DEVICES: the devices used by a user gives insight into their income and/or how they spend money
- MERCHANT_MAP: merchants of a certain categories
- CATEGORIES_WITH_WEIGHTS: percent of income spent on a category
- AMOUNT_RANGES: a realistic range to how much is spent on a category in rupees (salary handled casewise later)


In [11]:
INDIAN_CITIES = [
    "Mumbai","Delhi","Bengaluru","Hyderabad","Chennai","Kolkata",
    "Pune","Ahmedabad","Jaipur","Lucknow","Patna","Bhopal",
    "Nagpur","Indore","Coimbatore","Mysuru","Vadodara","Surat","Ludhiana","Agra"
]
CITY_TIER = {
    "Mumbai":1,"Delhi":1,"Bengaluru":1,"Hyderabad":1,"Chennai":1,
    "Kolkata":1,"Pune":1,"Ahmedabad":1,"Jaipur":2,"Lucknow":2,
    "Nagpur":2,"Indore":2,"Coimbatore":2,"Mysuru":2,"Vadodara":2,
    "Surat":2,"Patna":3,"Bhopal":3,"Ludhiana":3,"Agra":3,
}
DEVICES  = ["Android-Budget","Android-Mid","Android-Flagship","iPhone"]
UPI_APPS = ["PhonePe","GPay","Paytm","BHIM","AmazonPay"]
BANKS    = ["SBI","HDFC","ICICI","Axis","Kotak","PNB","BOB","Canara"]
GENDERS  = ["M","F","Other"]
STATES   = ["MH","DL","KA","TN","UP","WB","GJ","RJ","MP","TG"]

MERCHANT_MAP = {
    "groceries":       ["DMart","BigBasket","Reliance Fresh","Local Kirana","Spencer's"],
    "fuel":            ["BPCL","HPCL","Indian Oil","Shell","Reliance Petro"],
    "emi":             ["Bajaj Finserv","HDFC EMI","ZestMoney","LazyPay","Home Credit"],
    "entertainment":   ["BookMyShow","Zomato","Swiggy","Netflix","Hotstar"],
    "utilities":       ["BESCOM","MSEB","Tata Power","Jio","Airtel"],
    "rent":            ["NoBroker","Direct Landlord","MagicBricks","99acres","Housing"],
    "salary":          ["Employer NEFT","Salary Credit","Stipend","Freelance Payment","Consultancy Fee"],
    "postpaid_upgrade":["Jio Postpaid","Airtel Postpaid","Vi Postpaid","BSNL Postpaid"],
    "transfer":        ["Self Transfer","Family","Friend","Split Bill","Refund"],
    "insurance":       ["LIC","HDFC Life","Star Health","Bajaj Allianz","SBI Life"],
    "investment":      ["Zerodha","Groww","Upstox","SBI MF","Paytm Money"],
}

CATEGORIES_WITH_WEIGHTS = [
    ("groceries",25),("fuel",12),("emi",8),("entertainment",18),
    ("utilities",12),("rent",8),("transfer",10),("insurance",4),("investment",3),
]
CAT_NAMES, CAT_WEIGHTS = zip(*CATEGORIES_WITH_WEIGHTS)

AMOUNT_RANGES = {
    "groceries":(150,  3000),
    "fuel":(200,  3500),
    "emi":(1000, 18000),
    "entertainment":(50,   2000),
    "utilities":(300,  4000),
    "rent":(3000, 30000),
    "transfer":(100,  50000),
    "insurance":(500,  5000),
    "investment":(500,  25000),
    "salary":(0, 0),          
    "postpaid_upgrade":(299, 999),
}

print("reference defined")

reference defined


### 1.2. Generate sample user

In [12]:
def generate_user(user_id):
    archetype_name= random.choice(list(ARCHETYPES.keys()))
    arch= ARCHETYPES[archetype_name]
    city= random.choice(INDIAN_CITIES)

    return {
        "user_id":user_id,
        "archetype":archetype_name,
        "age":random.randint(18, 60),
        "gender":random.choice(GENDERS),
        "city":city,
        "state":random.choice(STATES),
        "city_tier":CITY_TIER[city],
        "primary_upi_app":random.choice(UPI_APPS),
        "device_type":random.choice(DEVICES),
        "primary_bank":random.choice(BANKS),
        "num_linked_accounts":random.randint(1, 4),
        "account_age_months":random.randint(3, 72),
        "base_monthly_income":random.randint(*arch["income_range"]),
        "spending_ratio":random.uniform(*arch["spending_ratio"]),
        "salary_day":random.randint(1, 5) if archetype_name == "salaried_stable" else random.randint(1, 28),
        "bill_regularity":arch["bill_regularity"] + random.uniform(-0.05, 0.05),
        "gig_variance":arch["gig_variance"],
        "default_risk":arch["default_risk"],
        "went_postpaid":random.random() < 0.40, # 40% upgraded prepaid → postpaid
        "has_investment_account":random.random() < 0.30,
        "kyc_verified":random.random() < 0.85,
    }

sample_user= generate_user("USR_TEST")
pd.Series(sample_user)

user_id                      USR_TEST
archetype                     student
age                                39
gender                              M
city                           Bhopal
state                              DL
city_tier                           3
primary_upi_app                  GPay
device_type               Android-Mid
primary_bank                   Canara
num_linked_accounts                 2
account_age_months                 67
base_monthly_income              2126
spending_ratio               1.010144
salary_day                         25
bill_regularity              0.627289
gig_variance                      0.5
default_risk                     0.25
went_postpaid                   False
has_investment_account          False
kyc_verified                    False
dtype: object

### 1.3. Transaction stream generator

12 months of UPI transaction history for each user. Key decisions (asked AI for possible unique factors):

- **income noise**: gig workers have high variance month-to-month (`gig_variance`)
- **festival season**: October (Diwali) and March (year-end) spending jumps 30%
- **salary day**: salary arrives ±2 days around the expected date (realistic)
- **failure rate**: riskier archetypes fail transactions more often

In [13]:
def generate_transactions(user, num_months=12):
    txns  = []
    start = datetime(2023, 1, 1)

    for m in range(num_months):
        month_start = start + timedelta(days=30 * m)
        cur_month   = month_start.month

        #income noise
        noise          = max(np.random.normal(1.0, user["gig_variance"]), 0.2)
        monthly_income = user["base_monthly_income"] * noise

        #festival months
        fest_mult = 1.3 if cur_month in [10, 3] else 1.0

        #salary credit
        sal_date = month_start + timedelta(days=user["salary_day"] + random.randint(-2, 2))
        txns.append({
            "user_id":user["user_id"], "date": sal_date,
            "amount":round(monthly_income, 2), "type": "inflow",
            "category":"salary", "merchant": random.choice(MERCHANT_MAP["salary"]),
            "upi_app":user["primary_upi_app"], "bank": user["primary_bank"],
            "device":user["device_type"], "city": user["city"],
            "failed":False, "is_recurring": True, "month": m + 1,
        })

        for _ in range(random.randint(8, 30)):
            cat = random.choices(CAT_NAMES, weights=CAT_WEIGHTS)[0]

            if cat == "investment" and not user["has_investment_account"]:
                cat = "transfer"

            txns.append({
                "user_id":user["user_id"],
                "date":month_start + timedelta(days=random.randint(0, 29)),
                "amount":round(random.randint(*AMOUNT_RANGES[cat]) * fest_mult, 2),
                "type":"outflow",
                "category":cat,
                "merchant":random.choice(MERCHANT_MAP[cat]),
                "upi_app":random.choice(UPI_APPS), # people use multiple apps
                "bank":user["primary_bank"],
                "device":user["device_type"],
                "city":user["city"],
                # Riskier users fail more transactions
                "failed":random.random() < (user["default_risk"] * 0.25),
                "is_recurring":cat in ["emi", "rent", "utilities", "insurance"],
                "month":m + 1,
})

        if random.random() < user["bill_regularity"]:
            txns.append({
                "user_id":user["user_id"],
                "date":month_start + timedelta(days=random.randint(25, 30)),
                "amount":round(random.uniform(200, 1500), 2),
                "type":"outflow",
                "category":"utilities",
                "merchant":random.choice(MERCHANT_MAP["utilities"]),
                "upi_app":user["primary_upi_app"],
                "bank":user["primary_bank"],
                "device":user["device_type"],
                "city":user["city"],
                "failed":False,
                "is_recurring":True,
                "month":m + 1,
    })

    return txns

print("transaction generator defined")

transaction generator defined


### 1.4. Edge Cases

Some other unique factors related to life events (also used AI to advice on this):

1.**job loss**: 10% of users lose income for 2-3 months mid-year
2.**inactivity**: 8% go completely quiet for 1–2 months causing suspicion
3.**postpaid upgrade**: users who switched from prepaid mobile plan to postpaid, could indicate increasing income

In [14]:
def apply_edge_cases(txns, user):
    modified = []

    #job loss
    job_loss       = random.random() < 0.10
    job_loss_start = random.randint(4, 9)  if job_loss       else None

    #inactivity
    has_inactivity = random.random() < 0.08
    inactive_month = random.randint(2, 10) if has_inactivity else None

    for t in txns:
        m = t["month"]

        #dropping transactions during inactive time
        if has_inactivity and inactive_month <= m <= inactive_month + 1:
            continue

        #slash salary during job loss months
        if job_loss and job_loss_start <= m <= job_loss_start + 2:
            if t["category"] == "salary":
                t["amount"] = round(t["amount"] * random.uniform(0.1, 0.4), 2)

        modified.append(t)

    #postpaid upgrade is a positive factor
    if user["went_postpaid"]:
        upgrade_date = datetime(2023, 1, 1) + timedelta(days=random.randint(30, 300))
        modified.append({
            "user_id":user["user_id"], "date": upgrade_date,
            "amount": random.randint(299, 999), "type": "outflow",
            "category":"postpaid_upgrade",
            "merchant":random.choice(MERCHANT_MAP["postpaid_upgrade"]),
            "upi_app":user["primary_upi_app"], "bank": user["primary_bank"],
            "device":user["device_type"], "city": user["city"],
            "failed":False, "is_recurring": False, "month": upgrade_date.month,
        })

    return modified

print("edge cases defined")

edge cases defined


### 1.5. Final generation of 70000 users

In [15]:
NUM_USERS=70000
print(f"generating {NUM_USERS} users")

users_list=[generate_user(f"USR_{i:04d}") for i in range(NUM_USERS)]
users_df=pd.DataFrame(users_list)

all_txns=[]
for u in users_list:
    raw=generate_transactions(u)
    final=apply_edge_cases(raw, u)
    all_txns.extend(final)

txns_df=pd.DataFrame(all_txns)
txns_df["date"]=pd.to_datetime(txns_df["date"])
txns_df=txns_df.sort_values(["user_id", "date"]).reset_index(drop=True)

#derived columns
txns_df["day_of_week"]=txns_df["date"].dt.day_name()
txns_df["is_weekend"]=txns_df["day_of_week"].isin(["Saturday", "Sunday"])
txns_df["is_festival_month"]=txns_df["date"].dt.month.isin([10, 3])
txns_df["txn_size_bucket"]=pd.cut(
    txns_df["amount"], bins=[0, 500, 2000, 10000, 999999],
    labels=["micro", "small", "medium", "large"]
)

print(f"transactions:{txns_df.shape[0]:,} rows × {txns_df.shape[1]} columns")
print(f"users:{users_df.shape[0]:,} rows × {users_df.shape[1]} columns")
txns_df.head()

generating 70000 users
transactions: 17,246,203 rows × 17 columns
users: 70,000 rows × 21 columns


,user_id,date,amount,type,category,merchant,upi_app,bank,device,city,failed,is_recurring,month,day_of_week,is_weekend,is_festival_month,txn_size_bucket
0,USR_0000,2023-01-03,2989.0,outflow,fuel,BPCL,PhonePe,PNB,Android-Mid,Pune,False,False,1,Tuesday,False,False,medium
1,USR_0000,2023-01-05,2814.0,outflow,emi,Bajaj Finserv,Paytm,PNB,Android-Mid,Pune,False,True,1,Thursday,False,False,medium
2,USR_0000,2023-01-05,419.0,outflow,groceries,Reliance Fresh,GPay,PNB,Android-Mid,Pune,False,False,1,Thursday,False,False,micro
3,USR_0000,2023-01-06,1437.0,outflow,fuel,Indian Oil,AmazonPay,PNB,Android-Mid,Pune,False,False,1,Friday,False,False,small
4,USR_0000,2023-01-06,1968.0,outflow,utilities,Airtel,Paytm,PNB,Android-Mid,Pune,False,True,1,Friday,False,False,small


---
### 2. Feature Engineering

- **basic aggr.**: total spend, avg income
- **behavioral ratios**: savings rate, failure rate
- **rolling window**: historical averages, to compare with recent ones
- **consistency**


In [9]:
inflows=txns_df[txns_df["type"] == "inflow"]
outflows=txns_df[txns_df["type"] == "outflow"]
print(f"Inflow rows:{len(inflows):,}")
print(f"Outflow rows:{len(outflows):,}")

Inflow rows:828,878
Outflow rows:16,414,557


### 2.1. Basic Aggregations

simple summaries of 
- total money in
- total money out
- no. of transactions
- averages
- unique attributes

In [ ]:
total_inflow= inflows.groupby("user_id")["amount"].sum().rename("total_inflow_12m")
total_outflow= outflows.groupby("user_id")["amount"].sum().rename("total_outflow_12m")
avg_monthly_inflow= (total_inflow  / 12).rename("avg_monthly_inflow")
avg_monthly_outflow= (total_outflow / 12).rename("avg_monthly_outflow")
txn_count= txns_df.groupby("user_id")["amount"].count().rename("total_txn_count")
unique_merchants= txns_df.groupby("user_id")["merchant"].nunique().rename("unique_merchants")
unique_apps= txns_df.groupby("user_id")["upi_app"].nunique().rename("unique_upi_apps")

print("basic aggregations made: ")
pd.DataFrame({
    "total_inflow_12m": total_inflow.head(5),
    "avg_monthly_inflow": avg_monthly_inflow.head(5),
    "total_txn_count": txn_count.head(5),
})

### 2.2. Behavioral ratios

key features:
- **failure_rate**: % of transactions that failed, could indicate financial stress if high
- **savings_rate**: (inflow-outflow)/inflow, negative implies more spending than earning
- **recurring_payment_ratio**: % of spending on predictable needs like rent and emi, higher could imply more discipline and consistency
- **essential vs discretionary**: to check if there's frivolous spending

In [ ]:
#failure rate
total_txns_per_user  = txns_df.groupby("user_id")["failed"].count()
failed_txns_per_user = txns_df.groupby("user_id")["failed"].sum()
failure_rate         = (failed_txns_per_user / total_txns_per_user).rename("failure_rate")

#savings rate (spendings more than earnings indicated by negative)
savings_rate = ((total_inflow - total_outflow) / total_inflow.replace(0, np.nan)).rename("savings_rate")

#recurring payment ratio (eg:rent, emi) as % of outflow
recurring_outflows= outflows[outflows["is_recurring"] == True]
total_recurring= recurring_outflows.groupby("user_id")["amount"].sum()
recurring_ratio= (total_recurring / total_outflow.replace(0, np.nan)).rename("recurring_payment_ratio")

#category spend ratios:each category as % of outflow
category_spend= (
    outflows.groupby(["user_id", "category"])["amount"]
    .sum().unstack(fill_value=0)
)
for col in category_spend.columns:
    category_spend[col]= category_spend[col] / total_outflow.replace(0, np.nan)
category_spend.columns= [f"spend_ratio_{c}" for c in category_spend.columns]

#essential vs discretionary split
essential_cats= ["groceries", "utilities", "rent", "emi", "insurance"]
discretionary_cats= ["entertainment", "fuel", "transfer", "investment"]
essential_spend= outflows[outflows["category"].isin(essential_cats)].groupby("user_id")["amount"].sum()
disc_spend= outflows[outflows["category"].isin(discretionary_cats)].groupby("user_id")["amount"].sum()
essential_ratio= (essential_spend / total_outflow.replace(0, np.nan)).rename("essential_spend_ratio")
disc_ratio= (disc_spend      / total_outflow.replace(0, np.nan)).rename("discretionary_spend_ratio")

#binary signals: yes/no flags for insurance and investment
has_investments= (outflows[outflows["category"]=="investment"].groupby("user_id")["amount"].sum()>0).astype(int).rename("has_investment_activity")
has_insurance= (outflows[outflows["category"]=="insurance"].groupby("user_id")["amount"].sum()>0).astype(int).rename("has_insurance_payments")

print("behavioral ratios made: ")
pd.DataFrame({"failure_rate": failure_rate.head(5), "savings_rate": savings_rate.head(5)})

### 2.3. Rolling window

every user gets 12 months of history, to summarize and compare further, dividing it into historical (past 10 months) and recent (past 60 days)

-`income_trend_ratio`: say < 0.8 then the income is 20%+ below historical average which is a bad sign
-`spending_velocity_ratio`: say > 1.25 spending accelerating while income drops which is also a bad sign

In [16]:
max_date    = txns_df["date"].max()
recent_mask = txns_df["date"] >= max_date - pd.Timedelta(days=60)

recent_txns     = txns_df[recent_mask]
historical_txns = txns_df[~recent_mask]

#recent inflow vs historical average monthly inflow
recent_inflow = (recent_txns[recent_txns["type"]=="inflow"].groupby("user_id")["amount"].sum() / 2).rename("avg_inflow_last_60d")
hist_inflow   = (historical_txns[historical_txns["type"]=="inflow"].groupby("user_id")["amount"].sum() / 10).rename("avg_inflow_prev_10m")

#income_trend_ratio: > 1 if income improving, < 1 if income declining
income_trend  = (recent_inflow / hist_inflow.replace(0, np.nan)).rename("income_trend_ratio")

#recent outflow vs historical outflow
recent_outflow = (recent_txns[recent_txns["type"]=="outflow"].groupby("user_id")["amount"].sum() / 2).rename("avg_outflow_last_60d")
hist_outflow   = (historical_txns[historical_txns["type"]=="outflow"].groupby("user_id")["amount"].sum() / 10).rename("avg_outflow_prev_10m")

#spending_velocity_ratio: > 1 if spending more recently than historically
spend_velocity   = (recent_outflow / hist_outflow.replace(0, np.nan)).rename("spending_velocity_ratio")
recent_fail_rate = recent_txns.groupby("user_id")["failed"].mean().rename("failure_rate_last_60d")

print("rolling window made: ")
income_trend.describe().round(2)

KeyboardInterrupt: 

### 2.4. Consistency scores

consistent and predictable behavior is preferred as it means they'd pay on time
using the **coeff of variation (CV) = std/mean**. Lower CV=more consistent behaviour.
converting it to a 0–1 score where higher is more stable

**income_stability_score = 1 / (1 + CV)**

- CV near 0: stable income=score near 1.0
- CV near 1: chaotic income=score near 0.5

In [ ]:
#monthly inflow grouped by user and month
monthly_inflow_df = (txns_df[txns_df["type"] == "inflow"].groupby(
    ["user_id", "month"])["amount"].sum().reset_index())

#income stability: variation in **monthly** income
income_stats = monthly_inflow_df.groupby("user_id")["amount"].agg(
    ["mean", "std"])
income_cv = income_stats["std"] / income_stats["mean"].replace(0, np.nan)
income_stability = (1 / (1 + income_cv)).rename("income_stability_score")

#month by month income change volatility
m_sorted = monthly_inflow_df.sort_values(["user_id", "month"])
m_sorted["pct_change"] = m_sorted.groupby("user_id")["amount"].pct_change()
income_volatility = m_sorted.groupby("user_id")["pct_change"].std().rename(
    "income_volatility")

#bill payment consistency: months with utility payment in outflow
bill_consistency = (
    txns_df[(txns_df["category"] == "utilities")
            & (txns_df["type"] == "outflow")].groupby(["user_id", "month"])
    ["amount"].count().reset_index().groupby("user_id")["month"].count() /
    12).rename("bill_payment_consistency")

#salary day regularity (salaried individuals get paid predictably)
salary_dates = inflows[inflows["category"] == "salary"].copy()
salary_dates["day_of_month"] = salary_dates["date"].dt.day
salary_day_std = salary_dates.groupby("user_id")["day_of_month"].std().fillna(
    15)
salary_regularity = (1 / (1 + salary_day_std)).rename("salary_day_regularity")

#transaction frequency consistency: is there consistent transaction every month
txns_per_month = txns_df.groupby([
    "user_id", "month"
])["amount"].count().reset_index().groupby("user_id")["amount"].agg(
    ["mean", "std"])
txn_freq_cv = txns_per_month["std"] / txns_per_month["mean"].replace(0, np.nan)
txn_freq_consist = (1 / (1 + txn_freq_cv)).rename("txn_frequency_consistency")

#active months: how many months had 1 transaction at least out of the 12
active_months = (txns_df.groupby(
    ["user_id", "month"])["amount"].count().reset_index().groupby("user_id")
                 ["month"].count().rename("active_months_out_of_12"))

print("consistency made: ")
income_stability.describe().round(3)


### 2.5 Final assembly of feature engineering



In [ ]:
feature_df = users_df.set_index("user_id").copy()

for s in [
    total_inflow, total_outflow, avg_monthly_inflow, avg_monthly_outflow,
    txn_count, unique_merchants, unique_apps,
    failure_rate, savings_rate, recurring_ratio,
    essential_ratio, disc_ratio, has_investments, has_insurance,
    recent_inflow, recent_outflow, hist_inflow, hist_outflow,
    income_trend, spend_velocity, recent_fail_rate,
    income_stability, income_volatility, bill_consistency,
    salary_regularity, txn_freq_consist, active_months,
]:
    feature_df = feature_df.join(s, how="left")

feature_df = feature_df.join(category_spend, how="left")
feature_df = feature_df.fillna(0).reset_index()

print(f"feature table: {feature_df.shape[0]} users × {feature_df.shape[1]} columns")
feature_df[["user_id", "archetype", "avg_monthly_inflow", "savings_rate",
            "failure_rate", "income_stability_score", "bill_payment_consistency"]].head()

---
## 3. Labeling

Must rely on proxy labeling as i don't have real data on whether or not the user has defaulted or not, hence will flag high behavioral stress as "default"

a signal is worth 1 stress point. 3 or more points is labeled as default.

In [ ]:
stress = pd.Series(0, index=feature_df.index)
stress += (feature_df["savings_rate"] < -0.10).astype(int)  # spending > income
stress += (feature_df["failure_rate"] > 0.08).astype(int)  # frequent failures
stress += (feature_df["failure_rate_last_60d"]> 0.10).astype(int)  # recent failure spike
stress += (feature_df["income_trend_ratio"]< 0.80).astype(int)  # income declining
stress += (feature_df["spending_velocity_ratio"]> 1.25).astype(int)  # spend accelerating
stress += (feature_df["bill_payment_consistency"]< 0.50).astype(int)  # poor bill habits
stress += (feature_df["active_months_out_of_12"]< 8.00).astype(int)  # inactivity gaps

feature_df["stress_score"] = stress
feature_df["default_label"] = (stress >= 3).astype(int)

n_default = feature_df["default_label"].sum()
print(f"label distribution:")
print(f"good(0): {len(feature_df)-n_default} ({(1-n_default/len(feature_df))*100:.1f}%)")
print(f"default (1): {n_default} ({n_default/len(feature_df)*100:.1f}%)")
print(f"\nDefault rate by archetype:")
print(feature_df.groupby("archetype")["default_label"].mean().sort_values(ascending=False).round(3))


---
## 4. Modeling

We train two models and blend them:

- **Model A: logistic regression**
- **Model B: XGBoost**

**AUC-ROC** is our main metric — it measures how well the model *ranks* risky users above safe ones. 0.5 = random, 1.0 = perfect.

### Prepare Features

In [ ]:
# Drop columns that cause data leakage (used to BUILD the label)
# or columns that wouldn't exist in a real-world scenario
DROP_COLS = [
    "user_id", "default_label", "stress_score",
    "savings_rate", "failure_rate", "failure_rate_last_60d",
    "income_trend_ratio", "spending_velocity_ratio",
    "bill_payment_consistency", "active_months_out_of_12",
    "default_risk", "bill_regularity", "gig_variance", "spending_ratio", "salary_day",
]
feature_cols = [c for c in feature_df.columns if c not in DROP_COLS]
df_model     = feature_df[feature_cols].copy()

# Encode categorical columns — ML models need numbers, not strings
le = LabelEncoder()
for col in ["archetype","gender","city","state","primary_upi_app","device_type","primary_bank"]:
    if col in df_model.columns:
        df_model[col] = le.fit_transform(df_model[col].astype(str))

X = df_model.values
y = feature_df["default_label"].values

# stratify=y ensures both splits have equal proportions of defaults
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Feature matrix: {X.shape[0]} users × {X.shape[1]} features")
print(f"Train: {len(X_train)}, Test: {len(X_test)}")
print(f"Default rate: {y.mean():.1%}")

### Model A — Logistic Regression

In [ ]:
# Logistic Regression is sensitive to feature scale so we standardise first
# StandardScaler makes every feature have mean=0, std=1
scaler         = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)   # use SAME scaler — important!

lr_model = LogisticRegression(
    class_weight="balanced",   # automatically handles more Good than Default users
    max_iter=1000,
    random_state=42,
    C=0.1,                     # regularisation strength — prevents overfitting
)
lr_model.fit(X_train_scaled, y_train)

lr_probs = lr_model.predict_proba(X_test_scaled)[:, 1]
lr_preds = lr_model.predict(X_test_scaled)
lr_auc   = roc_auc_score(y_test, lr_probs)

print(f"Logistic Regression AUC-ROC: {lr_auc:.4f}")
print(classification_report(y_test, lr_preds, target_names=["Good", "Default"]))

### Model B — XGBoost (or Random Forest fallback)

In [ ]:
if USE_XGB:
    print("Training XGBoost...")
    # scale_pos_weight corrects for class imbalance
    # if 9x more Good than Default, weight defaults 9x heavier
    scale_weight = (y_train == 0).sum() / (y_train == 1).sum()

    tree_model = xgb.XGBClassifier(
        n_estimators=300,      # number of trees
        max_depth=4,           # tree depth — deeper = more complex patterns
        learning_rate=0.05,    # how much each tree contributes — lower = more stable
        subsample=0.8,         # use 80% of rows per tree — reduces overfitting
        colsample_bytree=0.8,  # use 80% of features per tree
        scale_pos_weight=scale_weight,
        eval_metric="auc",
        random_state=42,
        verbosity=0,
    )
    tree_model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)
    model_b_name = "XGBoost"
else:
    print("Training Random Forest (XGBoost not installed)...")
    tree_model = RandomForestClassifier(
        n_estimators=300, max_depth=8,
        min_samples_leaf=5, class_weight="balanced",
        random_state=42, n_jobs=-1,
    )
    tree_model.fit(X_train, y_train)
    model_b_name = "Random Forest"

tree_probs = tree_model.predict_proba(X_test)[:, 1]
tree_preds = tree_model.predict(X_test)
tree_auc   = roc_auc_score(y_test, tree_probs)

print(f"\n{model_b_name} AUC-ROC: {tree_auc:.4f}")
print(classification_report(y_test, tree_preds, target_names=["Good", "Default"]))

### Cross-Validation

A single train/test split can be lucky or unlucky. 5-fold CV splits data 5 different ways and averages the results — much more honest estimate of real-world performance.

In [ ]:
cv_scores = cross_val_score(
    tree_model, X, y,
    cv=StratifiedKFold(n_splits=5),
    scoring="roc_auc"
)
print(f"5-fold CV AUC scores: {cv_scores.round(4)}")
print(f"Mean: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")

### Ensemble — Blend Both Models

LR and tree models make different kinds of errors. Blending reduces overall error. We weight the tree model higher (0.7) since it's more powerful.

In [ ]:
lr_probs_all   = lr_model.predict_proba(scaler.transform(X))[:, 1]
tree_probs_all = tree_model.predict_proba(X)[:, 1]

# 30% Logistic Regression + 70% XGBoost/RF
ensemble_probs = lr_probs_all * 0.3 + tree_probs_all * 0.7
print("Ensemble created ✅")

---
## Phase 5 — Credit Scores (300–900 Scale)

We convert the ensemble's probability of default into a CIBIL-style score.

**Key idea:** Low probability of default → High score (like CIBIL)

`score = 300 + (1 - pd) × 600`

Then we bucket into bands: A (Excellent) → E (Very Poor)

In [ ]:
def prob_to_score(prob, lo=300, hi=900):
    return ((1 - prob) * (hi - lo) + lo).round(0).astype(int)

def assign_band(s):
    if   s >= 750: return "A - Excellent"
    elif s >= 650: return "B - Good"
    elif s >= 550: return "C - Fair"
    elif s >= 450: return "D - Poor"
    else:          return "E - Very Poor"

feature_df["pd_logistic"]   = lr_probs_all
feature_df["pd_tree_model"] = tree_probs_all
feature_df["pd_ensemble"]   = ensemble_probs
feature_df["credit_score"]  = prob_to_score(pd.Series(ensemble_probs))
feature_df["credit_band"]   = feature_df["credit_score"].apply(assign_band)

print("Credit Score Statistics:")
print(feature_df["credit_score"].describe().round(1))
print("\nBand Distribution:")
print(feature_df["credit_band"].value_counts().sort_index())
print("\nAverage Score by Archetype:")
print(feature_df.groupby("archetype")["credit_score"].mean().sort_values(ascending=False).round(1))

### Save Outputs

In [ ]:
output_cols = [
    "user_id","archetype","age","city","credit_score","credit_band",
    "default_label","pd_ensemble","pd_logistic","pd_tree_model",
    "avg_monthly_inflow","income_stability_score",
    "recurring_payment_ratio","unique_merchants",
]
feature_df[output_cols].to_csv("upi_credit_scores.csv",    index=False)
feature_df.to_csv              ("upi_features_labeled.csv", index=False)
txns_df.to_csv                 ("upi_transactions.csv",     index=False)
users_df.to_csv                ("upi_users.csv",            index=False)

print("✅saved:")
print("upi_credit_scores.csv")
print("upi_features_labeled.csv")
print("upi_transactions.csv")
print("upi_users.csv")

---
## Phase 6 — Visualisations

Six plots covering score distribution, archetype breakdown, ROC curves, feature importance, and the confusion matrix.

In [ ]:
fig = plt.figure(figsize=(18, 12))
fig.suptitle("UPI Credit Scoring Model — Results", fontsize=16, fontweight="bold")
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.35)

# Plot 1: Credit score distribution
ax1 = fig.add_subplot(gs[0, 0])
ax1.hist(feature_df["credit_score"], bins=40, color="#4F86C6", edgecolor="white", linewidth=0.5)
ax1.axvline(feature_df["credit_score"].mean(), color="red", linestyle="--",
            label=f'Mean: {feature_df["credit_score"].mean():.0f}')
ax1.set_title("Credit Score Distribution")
ax1.set_xlabel("Credit Score")
ax1.set_ylabel("Users")
ax1.legend()

# Plot 2: Average score by archetype
ax2 = fig.add_subplot(gs[0, 1])
arch_scores = feature_df.groupby("archetype")["credit_score"].mean().sort_values()
bars = ax2.barh(arch_scores.index, arch_scores.values,
                color=["#E74C3C","#E67E22","#F1C40F","#2ECC71","#3498DB"])
for bar, val in zip(bars, arch_scores.values):
    ax2.text(val+2, bar.get_y()+bar.get_height()/2, f'{val:.0f}', va='center', fontsize=9)
ax2.set_title("Avg Credit Score by Archetype")
ax2.set_xlabel("Average Credit Score")

# Plot 3: Band pie chart
ax3 = fig.add_subplot(gs[0, 2])
band_counts = feature_df["credit_band"].value_counts().sort_index()
ax3.pie(band_counts.values,
        labels=[b.split(" - ")[1] for b in band_counts.index],
        autopct='%1.1f%%',
        colors=["#E74C3C","#E67E22","#F1C40F","#3498DB","#2ECC71"][:len(band_counts)],
        startangle=90)
ax3.set_title("Credit Band Distribution")

# Plot 4: ROC curves
ax4 = fig.add_subplot(gs[1, 0])
for name, probs in [("Logistic Regression", lr_probs), (model_b_name, tree_probs)]:
    fpr, tpr, _ = roc_curve(y_test, probs)
    ax4.plot(fpr, tpr, label=f"{name} (AUC={roc_auc_score(y_test, probs):.3f})", linewidth=2)
ax4.plot([0,1],[0,1], "k--", alpha=0.4, label="Random")
ax4.set_title("ROC Curves")
ax4.set_xlabel("False Positive Rate")
ax4.set_ylabel("True Positive Rate")
ax4.legend(fontsize=8)

# Plot 5: Feature importance
ax5 = fig.add_subplot(gs[1, 1])
importance = pd.Series(tree_model.feature_importances_, index=df_model.columns).sort_values(ascending=False).head(12)
ax5.barh(importance.index[::-1], importance.values[::-1], color="#4F86C6")
ax5.set_title(f"Top 12 Features ({model_b_name})")
ax5.set_xlabel("Importance")
ax5.tick_params(axis='y', labelsize=8)

# Plot 6: Confusion matrix
ax6 = fig.add_subplot(gs[1, 2])
cm   = confusion_matrix(y_test, tree_preds)
disp = ConfusionMatrixDisplay(cm, display_labels=["Good","Default"])
disp.plot(ax=ax6, colorbar=False, cmap="Blues")
ax6.set_title(f"{model_b_name} Confusion Matrix")

plt.savefig("credit_score_results.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ Saved: credit_score_results.png")

---
## 🎉 Done!

Your full UPI credit scoring pipeline is complete. Here's what you built:

| Phase | What | Output |
|---|---|---|
| 1 | Data simulation | 245k realistic UPI transactions |
| 2 | Feature engineering | 58 behavioural features per user |
| 3 | Proxy labeling | Default labels from stress signals |
| 4 | Modeling | LR + XGBoost ensemble, AUC ~0.90 |
| 5 | Credit scores | 300–900 CIBIL-style scores |
| 6 | Visualisations | ROC, importance, distributions |

### What to explore next
- **SHAP** — explain each individual user's score
- **Fairness audit** — check if the model discriminates by city tier or gender
- **Streamlit dashboard** — build an interactive scoring app
